In [1]:

import pennylane as qml
import numpy as np
import json

dev = qml.device("default.qubit", wires=1)

@qml.qnode(dev)
def circuit(op):
    qml.Hadamard(wires=0)  # Not needed, but let's keep the original intention of applying a gate
    op
    return qml.probs(wires=0)

def random_gate(p, q, r):
    ops = [qml.PauliX(0), qml.PauliY(0), qml.PauliZ(0), qml.Identity(0)]
    probs = [p, q, r, 1 - p - q - r]
    probabilities = [circuit(op) for op in ops]
    mixed_probabilities = np.sum([prob * probability for prob, probability in zip(probs, probabilities)], axis=0)
    return mixed_probabilities.tolist()

def run(test_case_input: str) -> str:
    ins = np.array(json.loads(test_case_input))
    outs = random_gate(*ins)
    return json.dumps(outs)

def check(solution_output: str, expected_output: str) -> None:
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)

    assert np.allclose(solution_output, expected_output, rtol=1e-4), "Not the correct probabilities"

test_cases = [
    ('[0.25, 0.25, 0.25]', '[0.5, 0.5]'),
    ('[0.125, 0.25, 0.2]', '[0.625, 0.375]')
]

for i, (input_, expected_output) in enumerate(test_cases):
    print(f"Running test case {i} with input '{input_}'...")

    try:
        output = run(input_)

    except Exception as exc:
        print(f"Runtime Error. {exc}")

    else:
        try:
            check(output, expected_output)
            print("Correct!")
        except AssertionError as exc:
            print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'.")


Running test case 0 with input '[0.25, 0.25, 0.25]'...
Correct!
Running test case 1 with input '[0.125, 0.25, 0.2]'...
Wrong Answer. Have: '[0.4999999999999999, 0.4999999999999999]'. Want: '[0.625, 0.375]'.
